# Great Expectations Utility File
> From my previous observations about the data, I made this file to check for the data that I observed.
> Initially all my bronze checks should fail, BUT
> If I apply my silver transforms correctly I will be able to pass all the checks.

In [0]:
%pip install great_expectations

In [0]:
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd
from itertools import chain

In [0]:
# ── ANSI color codes for notebook output ──────────────────────────────────────
GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
BOLD   = "\033[1m"
DIM    = "\033[2m"
RESET  = "\033[0m"

def _header(table_name):
    bar = "─" * 60
    print(f"\n{CYAN}{BOLD}{bar}{RESET}")
    print(f"{CYAN}{BOLD}  TABLE: {table_name.upper()}{RESET}")
    print(f"{CYAN}{DIM}{bar}{RESET}")

def _pass_line():
    print(f"  {GREEN}{BOLD}✔  ALL CHECKS PASSED{RESET}")

def _fail_line(check_name, column, reason, examples=None, expected=None):
    print(f"\n  {RED}{BOLD}✘  {check_name}{RESET}")
    print(f"     {DIM}Column   :{RESET} {YELLOW}{column}{RESET}")
    print(f"     {DIM}Problem  :{RESET} {reason}")
    if expected:
        print(f"     {DIM}Expected :{RESET} {expected}")
    if examples:
        ex_str = ", ".join([f"'{x}'" for x in examples[:5]])
        print(f"     {DIM}Found    :{RESET} {RED}{ex_str}{RESET}")

def _summary(passed_tables, failed_tables):
    total = len(passed_tables) + len(failed_tables)
    bar   = "═" * 60
    print(f"\n{BOLD}{bar}{RESET}")
    print(f"{BOLD}  VALIDATION SUMMARY  —  {total} tables{RESET}")
    print(f"{BOLD}{bar}{RESET}")
    print(f"  {GREEN}{BOLD}✔ Passed:{RESET} {len(passed_tables)}")
    for t in passed_tables:
        print(f"      {GREEN}·{RESET} {t}")
    print(f"\n  {RED}{BOLD}✘ Failed:{RESET} {len(failed_tables)}")
    for t in failed_tables:
        print(f"      {RED}·{RESET} {t}")
    print(f"{BOLD}{bar}{RESET}\n")


# ── HUMAN-READABLE FAILURE REASONS ───────────────────────────────────────────

_REASONS = {
    "ExpectColumnValuesToNotBeNull":     "Column contains NULL values — these should not be null after Silver transforms.",
    "ExpectColumnValuesToBeOfType":      "Column has the wrong data type — the Silver cast transform may not have run or try_cast silently failed.",
    "ExpectColumnValuesToBeInSet":       "Column contains values outside the allowed set — mapping or normalisation transform may be incomplete.",
    "ExpectColumnValuesToNotMatchRegex": "Column contains values that match a banned pattern — normalisation did not fully clean this column.",
    "ExpectColumnValuesToMatchRegex":    "Column contains values that do not match the expected format — date or string cast may have failed.",
    "ExpectColumnValuesToBeBetween":     "Column contains values outside the expected numeric range.",
}

def _reason(exp):
    return _REASONS.get(exp.__class__.__name__, "Unexpected validation failure.")

def _expected_hint(exp):
    if hasattr(exp, "type_"):
        return f"pandas dtype = {exp.type_}"
    if hasattr(exp, "value_set") and exp.value_set is not None:
        vals = [str(v) for v in list(exp.value_set)[:8]]
        suffix = " …" if len(exp.value_set) > 8 else ""
        return f"{{{', '.join(vals)}}}{suffix}"
    if hasattr(exp, "regex"):
        return f"regex: {exp.regex}"
    if hasattr(exp, "min_value") or hasattr(exp, "max_value"):
        lo = getattr(exp, "min_value", "−∞")
        hi = getattr(exp, "max_value", "+∞")
        return f"between {lo} and {hi}"
    return None


# ── MAIN VALIDATE FUNCTION ────────────────────────────────────────────────────

def validate_table(table_name, df):
    pdf = df.toPandas()
    context = gx.get_context()
    batch = context.data_sources.pandas_default.read_dataframe(pdf, asset_name=table_name)

    expectations  = []
    failed_checks = []

    # ── circuits ──────────────────────────────────────────────────────────────
    if table_name == "race_data_circuits":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="circuitid"),
            gxe.ExpectColumnValuesToNotBeNull(column="country"),
            gxe.ExpectColumnValuesToBeInSet(column="country", value_set=[
                "Australia", "Malaysia", "Bahrain", "Spain", "Monaco",
                "Turkey", "Canada", "United States", "France", "Germany",
                "Hungary", "Belgium", "Italy", "Singapore", "Japan",
                "China", "Brazil", "United Arab Emirates", "United Kingdom",
                "Argentina", "Mexico", "Portugal", "Morocco", "Netherlands",
                "Austria", "Sweden", "Switzerland", "South Africa",
                "South Korea", "Korea", "India", "Russia", "Azerbaijan",
                "Saudi Arabia", "Qatar", "Vietnam",
            ]),
        ]

    # ── constructor_results ───────────────────────────────────────────────────
    elif table_name == "race_data_constructor_results":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeInSet(column="status", value_set=["Disqualified", None]),
        ]

    # ── constructor_standings ─────────────────────────────────────────────────
    elif table_name == "race_data_constructor_standings":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            # nullable int becomes float64 in pandas
            gxe.ExpectColumnValuesToBeOfType(column="position", type_="float64"),
            # positionText is mostly numeric strings like "1","2" — only check no raw codes remain
            gxe.ExpectColumnValuesToNotMatchRegex(column="positionText", regex=r"^[RDENFW\-\\]$"),
        ]

    # ── constructors ──────────────────────────────────────────────────────────
    elif table_name == "race_data_constructors":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToNotBeNull(column="name"),
        ]

    # ── driver_standings ──────────────────────────────────────────────────────
    elif table_name == "race_data_driver_standings":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToBeOfType(column="position", type_="float64"),
            gxe.ExpectColumnValuesToNotMatchRegex(column="positionText", regex=r"^[RDENFW\-\\]$"),
        ]

    # ── drivers ───────────────────────────────────────────────────────────────
    elif table_name == "race_data_drivers":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="forename"),
            gxe.ExpectColumnValuesToNotBeNull(column="surname"),
            gxe.ExpectColumnValuesToMatchRegex(column="forename", regex=r"^\S.*\S$|^\S$"),
            gxe.ExpectColumnValuesToMatchRegex(column="surname",  regex=r"^\S.*\S$|^\S$"),
            gxe.ExpectColumnValuesToBeOfType(column="number", type_="float64"),
            gxe.ExpectColumnValuesToMatchRegex(column="dob", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── lap_times ─────────────────────────────────────────────────────────────
    elif table_name == "race_data_lap_times":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="lap"),
            gxe.ExpectColumnValuesToBeBetween(column="lap", min_value=1, max_value=100),
            # nullable int = float64 in pandas
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="int32"),
            gxe.ExpectColumnValuesToNotBeNull(column="milliseconds"),
        ]

    # ── pit_stops ─────────────────────────────────────────────────────────────
    elif table_name == "race_data_pit_stops":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="stop"),
            gxe.ExpectColumnValuesToNotBeNull(column="lap"),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
        ]

    # ── qualifying ────────────────────────────────────────────────────────────
    elif table_name == "race_data_qualifying":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeOfType(column="q1", type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="q2", type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="q3", type_="float64"),
        ]

    # ── races ─────────────────────────────────────────────────────────────────
    elif table_name == "race_data_races":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="date"),
            gxe.ExpectColumnValuesToNotBeNull(column="circuitid"),
            gxe.ExpectColumnValuesToMatchRegex(column="date",        regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="fp1_date",    regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="fp2_date",    regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="fp3_date",    regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="quali_date",  regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="sprint_date", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── results ───────────────────────────────────────────────────────────────
    elif table_name == "race_data_results":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="resultid"),
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            # older seasons had more than 20 cars
            gxe.ExpectColumnValuesToBeBetween(column="positionorder", min_value=1, max_value=40),
            gxe.ExpectColumnValuesToBeOfType(column="number",       type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="grid",         type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="position",     type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="rank",         type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
            # no raw single-letter codes remaining after mapping()
            gxe.ExpectColumnValuesToNotMatchRegex(column="positionText", regex=r"^[RDENFW\-\\]$"),
            # fastestlapspeed — lowercase column name
            gxe.ExpectColumnValuesToBeOfType(column="fastestlapspeed", type_="float64"),
        ]

    # ── seasons ───────────────────────────────────────────────────────────────
    elif table_name == "race_data_seasons":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="year"),
            # year is stored as DateType (yyyy-01-01) after seasons_specific()
            gxe.ExpectColumnValuesToMatchRegex(column="year", regex=r"^\d{4}-01-01$"),
        ]

    # ── sprint_results ────────────────────────────────────────────────────────
    elif table_name == "race_data_sprint_results":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeOfType(column="position",     type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="rank",         type_="float64"),
            gxe.ExpectColumnValuesToNotMatchRegex(column="positionText", regex=r"^[RDENFW\-\\]$"),
        ]

    # ── status ────────────────────────────────────────────────────────────────
    elif table_name == "race_data_status":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="statusid"),
            gxe.ExpectColumnValuesToNotBeNull(column="status"),
        ]

    # ── fatal accidents drivers ───────────────────────────────────────────────
    elif table_name == "race_events_fatal_accidents_drivers":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="driver"),
            gxe.ExpectColumnValuesToMatchRegex(column="date_of_accident", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── fatal accidents marshalls ─────────────────────────────────────────────
    elif table_name == "race_events_fatal_accidents_marshalls":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="name"),
            gxe.ExpectColumnValuesToMatchRegex(column="date_of_accident", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── red flags ─────────────────────────────────────────────────────────────
    elif table_name == "race_events_red_flags":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="race"),
            gxe.ExpectColumnValuesToBeInSet(column="resumed", value_set=[
                "Resumed", "Standing Start", "Rolling Start", "Not Resumed", None
            ]),
        ]

    # ── safety cars ───────────────────────────────────────────────────────────
    elif table_name == "race_events_safety_cars":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="race"),
            gxe.ExpectColumnValuesToNotBeNull(column="cause"),
            gxe.ExpectColumnValuesToNotBeNull(column="deployed"),
            gxe.ExpectColumnValuesToNotMatchRegex(column="cause", regex=r"(?i)accident[s/]|stranded car/"),
        ]

    # ── GLOBAL CHECKS ─────────────────────────────────────────────────────────
    _header(table_name)

    if "url" in pdf.columns:
        failed_checks.append(("GlobalCheck", "url", "url column still present — drop_url() may not have run", None, None))

    if table_name == "race_data_races" and "year" in pdf.columns:
        failed_checks.append(("GlobalCheck", "year", "year column still present in races — races_specific() may not have run", None, None))

    # ── RUN EXPECTATIONS ──────────────────────────────────────────────────────
    for exp in expectations:
        try:
            result   = batch.validate(expect=exp)
            col_name = getattr(exp, "column", "Table-Level")
            if not result.success:
                stats    = result.result
                examples = stats.get("partial_unexpected_list", [])
                failed_checks.append((
                    exp.__class__.__name__,
                    col_name,
                    _reason(exp),
                    examples or None,
                    _expected_hint(exp),
                ))
        except Exception as e:
            failed_checks.append((
                exp.__class__.__name__,
                getattr(exp, "column", "unknown"),
                f"Execution error — {str(e)}",
                None,
                None,
            ))

    # ── PRINT RESULTS ─────────────────────────────────────────────────────────
    if not failed_checks:
        _pass_line()
    else:
        for (check, col, reason, examples, expected) in failed_checks:
            _fail_line(check, col, reason, examples, expected)

    passed = len(failed_checks) == 0
    return passed, [f"{c} on '{col}': {r}" for (c, col, r, _, _) in failed_checks]

In [0]:
# ── RUNNER — call this from your silver notebook ──────────────────────────────

def run_all_validations(schema="silver"):
    tables = spark.catalog.listTables(f"formone.{schema}")
    passed_tables = []
    failed_tables = []

    for table in tables:
        df = spark.table(f"formone.{schema}.{table.name}")
        ok, _ = validate_table(table.name, df)
        if ok:
            passed_tables.append(table.name)
        else:
            failed_tables.append(table.name)

    _summary(passed_tables, failed_tables)
    return passed_tables, failed_tables